# Rotations via Phase Gradients

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from sympy import Symbol
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

def show_signatures(*ops):
    display(SVG(field_signature_svg([(name, op.signature, {}) for name, op in ops])))

def show_calls(op):
    graph = op.call_graph()
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)

## `PhaseGradientUnitary`
Implementation of (Controlled-/)PhaseGradient unitary gate on an n-bit register.

The class supports implementing the phase gradient unitary gate and a controlled version
thereof. The n bit phase gradient unitary is defined as

$$
    \text{PhaseGrad}_{n, t} = \sum_{k=0}^{2^{n}-1}|k\rangle \langle k| \omega_{n, t}^{k}
$$

where

$$
    \omega_{n, t} = \exp\left(\frac{2\pi i t}{2^n}\right)
$$

The implementation simply decomposes into $n$ (controlled-) rotations, one on each qubit.

#### Registers
 - `phase_grad`: A THRU register which the phase gradient is applied to.
 - `(optional) ctrl`: A THRU register which specifies the control for this gate. Must have `is_controlled` set to `True` to use this register. 

#### Parameters
 - `bitsize`: The number of qubits of the register being acted on
 - `exponent`: $t$ in the above expression for $\omega_{n, t}$, a multiplicative factor for the phases applied to each state. Defaults to 1.0.
 - `is_controlled`: `bool` which determines if the unitary is controlled via a `ctrl` register.
 - `eps`: The precision for the total unitary, each underlying rotation is synthesized to a precision of `eps` / `bitsize`. 

Costs:
    qubits: 0 ancilla qubits are allocated.
    T-gates: Only uses 1 T gate explicitly but does rely on more costly Z rotations.
    rotations: Uses $n$ rotations with angles varying from 1/2 (for a single T-gate) to 1/(2^n).

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Appendix A: Addition for controlled rotations
 - [Halving the cost of quantum addition](https://arxiv.org/abs/1709.06648). Gidney (2017).


In [ ]:
from examples.algorithms.phase_gradient_catalog import GradientUnitaryExample

### Example Instances

In [ ]:
unitary = GradientUnitaryExample()

In [ ]:
n = Symbol("n")
symbolic_unitary = GradientUnitaryExample(n)

#### Graphical Signature

In [ ]:
show_signatures(("Four-bit gradient unitary", unitary), ("Symbolic gradient unitary", symbolic_unitary))

### Call Graph

In [ ]:
show_calls(unitary)
display(symbolic_unitary.call_graph().leaf_counts, symbolic_unitary.resource_model())

## `PhaseGradientState`
Prepare a phase gradient state $|\phi\rangle$ on a new register of bitsize $b_{grad}$

$$
    |\phi\rangle = \frac{1}{\sqrt{2^{n}}} \sum_{k=0}^{2^{n} - 1} \omega_{n, t}^{k} |k\rangle
$$

where

$$
    \omega_{n, t} = \exp\left(\frac{2\pi i t}{2^n}\right)
$$

Allocates / deallocates registers to store the phase gradient state and delegates
to the `PhaseGradientUnitary` bloq defined above.

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Appendix A: Addition for controlled rotations


In [ ]:
from examples.algorithms.phase_gradient_catalog import GradientStateExample

### Example Instances

In [ ]:
state = GradientStateExample()

#### Graphical Signature

In [ ]:
show_signatures(("Four-bit gradient preparation", state))

### Call Graph

In [ ]:
show_calls(state)

## `AddIntoPhaseGrad`
Quantum-quantum addition into a phase gradient register using $b_{phase} - 2$ Toffolis

$$
    U|x\rangle|\text{phase\_grad}\rangle = |x\rangle|\text{phase\_grad} + x\rangle
$$

#### Parameters
 - `x_bitsize`: Size of input register.
 - `phase_bitsize`: Size of phase gradient register to which the input value should be added.
 - `right_shift`: An integer specifying the amount by which the input register x should be right shifted before adding to the phase gradient register.
 - `sign`: Whether the input register x should be  added or subtracted from the phase gradient register.
 - `controlled_by`: Whether to control this bloq with a ctrl register. When controlled_by=None, this bloq is not controlled. When controlled_by=0, this bloq is active when the ctrl register is 0. When controlled_by=1, this bloq is active when the ctrl register is 1. 

#### Registers
 - `- ctrl`: Control THRU register
 - `- x`: Input THRU register storing input value x to be added to the phase gradient register.
 - `- phase_grad`: Phase gradient THRU register. 

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Appendix A: Addition for controlled rotations


In [ ]:
from examples.algorithms.phase_gradient_catalog import GradientAdditionExample

### Example Instances

In [ ]:
addition = GradientAdditionExample()

#### Graphical Signature

In [ ]:
show_signatures(("Four-bit gradient addition", addition))

### Call Graph

In [ ]:
show_calls(addition)

## `AddScaledValIntoPhaseReg`
Optimized quantum-quantum addition into a phase gradient register scaled by a constant $\gamma$.

$$
    U(\gamma)|x\rangle|\text{phase_grad}\rangle = |x\rangle|\text{phase_grad} + x * \gamma\rangle
$$

The operation calls `AddIntoPhaseGrad` gate $(\text{gamma_bitsize} + 2) / 2$ times.

#### Parameters
 - `x_dtype`: Fixed point specification of the input register.
 - `phase_bitsize`: Size of phase gradient register to which the scaled input should be added.
 - `gamma`: Floating point scaling factor.
 - `gamma_dtype`: `QFxp` data type capturing number of bits of precisions to be used for integer and fractional part of `gamma`. 

#### Registers
 - `x`: Input THRU register storing input value x to be scaled and added to the phase gradient register.
 - `phase_grad`: Phase gradient THRU register. 

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Appendix A: Addition for controlled rotations


In [ ]:
# The scaled example uses the same register binding as plain addition.

### Example Instances

In [ ]:
scaled = GradientAdditionExample(scaled=True)

#### Graphical Signature

In [ ]:
show_signatures(("Two-bit scaled gradient addition", scaled))

### Call Graph

In [ ]:
show_calls(scaled)

### Append the concrete operations

Use big-endian fixed-point registers. State preparation starts from a zero gradient word. Addition preserves the input and updates an arbitrary gradient word modulo its width. The measured carries start and end at zero. The scaled example retains the source's two-bit coefficient format, which saturates the literal coefficient 2 to 0.75. The source graph declares logical costs, and the native circuit reports emitted gates and measurements. The symbolic unitary can be modeled at width $n$ and needs a concrete width to append.

In [ ]:
for op, measurements in ((unitary, 0), (state, 0), (addition, 3), (scaled, 1)):
    circuit = DynamicCircuit(op.qubit_count)
    circuit.append(op)
    counts = Counter(gate.name for gate in circuit.to_ir().instructions)
    model = estimate_component_resources(op.resource_model())
    print(op.call_graph().root)
    print("Emitted on", op.qubit_count, "wires", dict(counts))
    print("Source logical resources", model)
    assert counts["measure"] == measurements